### Usage Metrics
We can now define the methodology of our investigation. First we will consider observable actions that can result in a possession ending. This can be a shot on goal, an inaccurate pass, a player being dispossessed, an unsuccessful dribble or a miscontrol. The sum of these will define a player's total number of Possession Ending Actions (PEA). This data was obtainable over the 25/26 season, with the exception of miscontrols. Therefore, a player's PEA will be calculated as such:
$$\text{PEA} = \text{Shots} + \text{Inaccurate Passes} + \text{Dispossessed} + \text{Unsuccessful Dribbles} $$
Since not all players across the league play an even amount of minutes, the PEA values are normalised for minutes played to create a player's PEA/90:
$$\text{PEA / 90} = \frac{\text{PEA}}{\text{Minutes Played}} \times 90 $$

Since this investigation aims to evaluate how well players use the ball given their share of PEA, we need a metric to define output. Shot Creating Actions (SCA) are the perfect metric, defined as the two offensive actions immediately proceeding a shot. This is a really useful metric since it credits secondary actions like an interception or take on which leads to a pass leading to a shot, therefore giving a better view of a player's offensive contribution. We have now been able to manually count SCA thanks to the soccerdata package which allowed us to scrape event data from all 380 games across the 2025/26 season. Using this data, we created an algorithm to manually identify each shot taken across the season and then identify the direct and secondary SCAs when applicable, storing their type as well. This will allow a better analysis how how well players use the ball than our previous analysis which relied on direct chances created. 

The limitation with the current framework is that it is hard to compare PEA / 90 and SCA / 90 across all teams in the league due to possession. Players on teams with greater average possession generally have more opportunity to become involved in possession ending actions and chance creation. For example, Rayan Cherki might have higher PEA / 90 and SCA / 90 stats than a player like Harry Wilson since it is likely that Manchester City will on average see a much higher share of possesion than Fulham so these stats are not immediately informative. We will therefore assume that player involvement scales linearly with team possesion and normalise so that each team has a neutral 50% possession share, adjusting the stats as such:
$$\text{PEA / 90}_{\text{adj}} = \text{PEA / 90} \times \frac{50}{\text{Team Possession}}$$
$$\text{SCA / 90}_{\text{adj}} = \text{SCA / 90} \times \frac{50}{\text{Team Possession}}$$

At this point we will be able to make a fair comparison of all players across the league and identify how SCA / 90 varies with PEA / 90 across different groups of positions.

To investigate a player's role within their own team, we want to look at how much of a team's possession flow through a single player and how much of the team's total output can be attributed to that player. Since we will not be considering goalkeepers in this analysis, the total team PEA and SCA will be calculated as the sum of all elgible outfield players:
$$\text{Team PEA} = \sum{\text{Total Player PEA}}$$
$$\text{Team SCA} = \sum{\text{Total Player SCA}}$$

Then we can define a new metric, a player's Usage Rate, which is the proportion of team possessions that end with that player:
$$\text{Usage Rate} = \frac{\text{Player PEA}}{\text{Team PEA}}$$
We can also find a player's Attacking Involvement as:
$$\text{Attacking Involvement} = \frac{\text{Player SCA}}{\text{Team SCA}}$$

Plotting these two metrics against each other, we will be able to identify a player's importance within their team and compare it to their output, creating one final metric, Relative Creation Efficiency (RCE):
$$\text{RCE} = \frac{\text{Attacking Involvement}}{\text{Usage Rate}}$$

From this analysis we will be able to measure the standout players across the league and also the most important players within specific teams.

In [2]:
# Import the cleaned data of players and teams
import pandas as pd
from pathlib import Path

# Data folders
project_folder = Path.cwd().parent 
cleaned_data_folder = project_folder / "data" / "cleaned"


players_df = pd.read_csv(cleaned_data_folder / "player_stats_cleaned.csv")
teams_df = pd.read_csv(cleaned_data_folder / "team_stats_cleaned.csv")
sca_df = pd.read_csv(cleaned_data_folder / "SCA.csv")

In [4]:
# 1: Create columns for PEA and PEA/90
players_df["PEA"] = players_df["totalShots"] + players_df["inaccuratePasses"] + players_df["dispossessed"] + players_df["unsuccessfulDribbles"]
players_df["PEA/90"] = (players_df["PEA"] / players_df["minutesPlayed"]) * 90

# Round PEA/90 
players_df["PEA/90"] = players_df["PEA/90"].round(2)

# Check 
print(players_df[["playerName","minutesPlayed","totalShots","inaccuratePasses", "dispossessed", "unsuccessfulDribbles","PEA","PEA/90"]].head())

        playerName  minutesPlayed  totalShots  inaccuratePasses  dispossessed  \
0  Bruno Fernandes           3066          85               355            30   
1      Declan Rice           3099          40               271            18   
2  Bruno Guimarães           2455          42               201            46   
3            Rodri           1513          16               160            11   
4      Jérémy Doku           1784          39               132            49   

   unsuccessfulDribbles  PEA  PEA/90  
0                    21  491   14.41  
1                    14  343    9.96  
2                    30  319   11.69  
3                     8  195   11.60  
4                    64  284   14.33  


In [78]:
# 2: Merge the player stats with the SCA data
# Check the number of players in each df
print(players_df["playerName"].nunique())
print(sca_df["player"].nunique())

# Check how many players are common between the dfs
common_players = set(players_df["playerName"]) & set(sca_df["player"])
print(len(common_players))

# Find closely matched player names which may differ because of accents, nicknames or punctuation
from difflib import get_close_matches

for player in missing_sca["playerName"]:
    match = get_close_matches(
        player,
        sca_df["player"].tolist(),
        n=1,
        cutoff=0.7
    )
    
    print(player, "->", match)

# Create a player name map
name_mapping = {
    "Josko Gvardiol" : "Joško Gvardiol",
    "Callum Hudson Odoi" : "Callum Hudson-Odoi",
    "Daniel Ballard" : "Dan Ballard",
    "Mateo Kovacic" : "Mateo Kovačić",
    "Idrissa Gueye" : "Idrissa Gana Gueye",
    "Ferdi Kadioglu" : "Ferdi Kadıoğlu",
    "Nikola Milenkovic" : "Nikola Milenković",
    "Junior Kroupi" : "Eli Junior Kroupi",
    "Benjamin Sesko" : "Benjamin Šeško",
    "Tomás Soucek" : "Tomáš Souček",
    "Ladislav Krejcí" : "Ladislav Krejčí",
    "Florentino" : "Florentino Luís",
    "Vitalii Mykolenko" : "Vitaliy Mykolenko",
    "Fábio Carvalho" : "Fabio Carvalho",
    "Sasa Lukic" : "Saša Lukić",
    "Jair Cunha" : "Jair",
    "Josh King" : "Joshua King",
    "Milos Kerkez" : "Miloš Kerkez",
    "Hwang Hee-Chan" : "Hwang Hee-chan",
    "Ollie Scarles" : "Oliver Scarles",
    "Maximilian Kilman" : "Max Kilman",
    "Radu Dragusin" : "Radu Drăguşin",
    "Hamed Traoré" : "Hamed Junior Traorè",
    "Charly Alcaraz" : "Carlos Alcaraz"}

# Apply player mapping
sca_df["player"] = sca_df["player"].replace(name_mapping)

# Merge the dataframes
merged_df = players_df.merge(
    sca_df,
    left_on="playerName",
    right_on="player",
    how="left",
    validate="one_to_one"
)
# Remove duplicate name column
merged_df = merged_df.drop(columns="player")

# Fill missing SCA stats with zero
merged_df["total_sca"] = merged_df["total_sca"].fillna(0).astype(int)

# Calculate SCA / 90
merged_df["SCA/90"] = ((merged_df["total_sca"] / merged_df["minutesPlayed"])*90).round(2)

496
481
447
Tom Edozie -> []
Joško Gvardiol -> ['Joško Gvardiol']
Jenson Seelt -> []
Callum Hudson Odoi -> ['Callum Hudson-Odoi']
Daniel Ballard -> ['Dan Ballard']
Mateo Kovačić -> ['Mateo Kovačić']
Idrissa Gana Gueye -> ['Idrissa Gana Gueye']
Ferdi Kadıoğlu -> ['Ferdi Kadıoğlu']
Nikola Milenković -> ['Nikola Milenković']
Eli Junior Kroupi -> ['Eli Junior Kroupi']
Dário Essugo -> []
Veljko Milosavljević -> ['Veljko Milosavljevic']
Benjamin Šeško -> ['Benjamin Šeško']
Tomáš Souček -> ['Tomáš Souček']
Ethan Pinnock -> []
Ladislav Krejčí -> ['Ladislav Krejčí']
Jun'ai Byfield -> []
Alex Murphy -> []
Florentino Luís -> ['Florentino Luís']
Vitaliy Mykolenko -> ['Vitaliy Mykolenko']
Fabio Carvalho -> ['Fabio Carvalho']
Saša Lukić -> ['Saša Lukić']
Jair -> ['Jair']
Joshua King -> ['Joshua King']
Miloš Kerkez -> ['Miloš Kerkez']
Odsonne Édouard -> []
Tyler Fredricson -> []
Jota Silva -> []
Jamaldeen Jimoh -> ['Jamaldeen Jimoh-Aloba']
Callum Olusesi -> []
Tyler Fletcher -> []
Tawanda Chirewa -> 

In [80]:
# 3: Create columns for possession adjusted PEA/90 and SCA/90
merged_df["adjPEA/90"] = (merged_df["PEA/90"] * (50 / merged_df["averageBallPossession"])).round(2)
merged_df["adjSCA/90"] = (merged_df["SCA/90"] * (50 / merged_df["averageBallPossession"])).round(2)

# Check
print(merged_df[["playerName","averageBallPossession","PEA/90","adjPEA/90","SCA/90","adjSCA/90"]].head())

        playerName  averageBallPossession  PEA/90  adjPEA/90  SCA/90  \
0  Bruno Fernandes                  51.79   14.41      13.91    5.90   
1      Declan Rice                  56.13    9.96       8.87    2.82   
2  Bruno Guimarães                  52.61   11.69      11.11    2.90   
3            Rodri                  60.50   11.60       9.59    2.56   
4      Jérémy Doku                  60.50   14.33      11.84    5.55   

   adjSCA/90  
0       5.70  
1       2.51  
2       2.76  
3       2.12  
4       4.59  


In [94]:
# 4: Create columns in the teams dataframe for total PEA and CC
# Obtain team total PEA and CC in a new df
team_totals = (merged_df.groupby("teamId")[["PEA","total_sca"]].sum().reset_index())

# Rename the columns
team_totals = team_totals.rename(columns = {
    "PEA" : "teamTotalPEA",
    "total_sca" : "teamTotalSCA"
})

# Merge the team total PEA and CC into the teams_df
teams_df = teams_df.merge(team_totals,
                         on = "teamId",
                         how = "left")

# Check
print(teams_df.head())

                 teamName  teamId  goalsScored  averageBallPossession  \
0                 Arsenal      42           71                  56.13   
1            Leeds United      34           49                  45.74   
2  Brighton & Hove Albion      30           52                  54.08   
3               Brentford      50           55                  47.61   
4              Sunderland      41           42                  44.84   

   totalPEA  total_sca_x  teamTotalPEA_x  total_sca_y  teamTotalPEA_y  \
0      3614          747            3614          747            3614   
1      3578          580            3578          580            3578   
2      3682          680            3682          680            3682   
3      3707          519            3707          519            3707   
4      3339          465            3339          465            3339   

   total_sca  teamTotalPEA  teamTotalSCA  
0        747          3614           747  
1        580          3578          

In [98]:
# 5: Calculate Usage Rates and Chance Creation Involvement 
# Remove existing team totals if they are already present (due to re-running the script)
merged_df = merged_df.drop(columns=["teamTotalPEA", "teamTotalSCA"],
                             errors="ignore")

# Merge the team totals into the players_dfplayers_df = players_df.merge(
merged_df = merged_df.merge(teams_df[["teamId", "teamTotalPEA", "teamTotalSCA"]],
                              on="teamId",
                              how="left")

# Calculate Usage rate
merged_df["usageRate"] = (merged_df["PEA"] / merged_df["teamTotalPEA"]).round(3)

# Calculate Chance Creation Involvement
merged_df["attackingInvolvement"] = (merged_df["total_sca"] / merged_df["teamTotalSCA"]).round(3)

# Check
print(merged_df[["playerName", "usageRate", "attackingInvolvement"]].head())

# Check that the usage rate and chance involvement roughly sums to 1 for each team allowing for slight deviation since usage rate and chance involvement were rounded to 3 decimal places
print(merged_df.groupby("teamId")["usageRate"].sum())
print(merged_df.groupby("teamId")["attackingInvolvement"].sum())

        playerName  usageRate  attackingInvolvement
0  Bruno Fernandes      0.131                 0.241
1      Declan Rice      0.095                 0.130
2  Bruno Guimarães      0.088                 0.121
3            Rodri      0.047                 0.047
4      Jérémy Doku      0.069                 0.120
teamId
3     0.998
6     1.002
7     1.001
14    1.001
17    1.001
30    1.001
33    0.998
34    1.003
35    0.998
37    1.000
38    1.000
39    1.001
40    0.999
41    1.000
42    1.001
43    0.999
44    0.998
48    0.999
50    1.003
60    1.001
Name: usageRate, dtype: float64
teamId
3     1.001
6     1.002
7     0.999
14    1.002
17    1.001
30    0.999
33    1.001
34    1.000
35    1.001
37    0.998
38    1.001
39    1.000
40    0.995
41    1.001
42    1.000
43    0.998
44    1.000
48    1.001
50    1.001
60    0.999
Name: attackingInvolvement, dtype: float64


In [102]:
# 6: Create one final metrics which is Relative Creation Efficiency (REC)
merged_df["RCE"] = (merged_df["attackingInvolvement"] / merged_df["usageRate"]).round(3)

# Convert inf values to nan
import numpy as np
merged_df["RCE"] = merged_df["RCE"].replace([np.inf, -np.inf], np.nan)

# Fill nan values
merged_df["RCE"] = merged_df["RCE"].fillna(0)

In [104]:
# 6: Export the final sheet 
# Define the list of columns
feature_cols = ["playerName","playerId","teamName","teamId","primaryPosition","positionGroup","minutesPlayed",
               "PEA","PEA/90","adjPEA/90",
               "total_sca", "SCA/90", "adjSCA/90",
               "usageRate", "attackingInvolvement", "RCE",
               "assists","expectedAssists","goals","expectedGoals",
               "marketValue"]
# Create a new df
player_usage_data = merged_df[feature_cols].copy()

# Export the final player usage metrics df as well as a master copy of the full data
player_usage_data.to_csv(project_folder / "data" / "player_usage_data.csv", index = False)
merged_df.to_csv(project_folder / "data" / "player_data_master.csv", index = False)